# 03 - Validation du pipeline

Procedure de validation en quatre volets : **coherence** entre couches, **conformite** des champs calcules, **stabilite** (generalisation du score sur deux moities disjointes du corpus) et **verification manuelle** par echantillon.

Chaque test affiche OK / ECHEC avec le detail.

In [ ]:
%pip install -q pandas pyarrow boto3 matplotlib requests "elasticsearch==8.13.2" scipy
import io, json, re, sys, collections
import boto3, pandas as pd, matplotlib.pyplot as plt, requests
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
sys.path.insert(0, "../src")          # modules de normalisation du projet (purs, sans dependance Docker)

# Acces MinIO depuis votre machine (la stack Docker doit tourner)
s3 = boto3.client("s3", endpoint_url="http://localhost:9000", aws_access_key_id="minioadmin",
                  aws_secret_access_key="minioadmin123", region_name="us-east-1")

def list_keys(bucket, prefix=""):
    out, token = [], None
    while True:
        kw = dict(Bucket=bucket, Prefix=prefix, **({"ContinuationToken": token} if token else {}))
        r = s3.list_objects_v2(**kw); out += [o["Key"] for o in r.get("Contents", [])]
        if not r.get("IsTruncated"): return out
        token = r["NextContinuationToken"]

def read_jsonl(bucket, keys):
    rows = []
    for k in keys:
        body = s3.get_object(Bucket=bucket, Key=k)["Body"].read().decode("utf-8")
        rows += [json.loads(l) for l in body.splitlines() if l]
    return pd.DataFrame(rows)

def latest_partition(bucket):
    keys = sorted(k for k in list_keys(bucket) if k.endswith(".jsonl") and k.startswith("dt="))
    return [k for k in keys if k.startswith(keys[-1].split("/")[0] + "/")]

def read_silver(bucket):
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=bucket, Key="data/part-0000.parquet")["Body"].read()))

def read_report(bucket, key="_reports/latest.json"):
    return json.loads(s3.get_object(Bucket=bucket, Key=key)["Body"].read())

from elasticsearch import Elasticsearch, helpers
es = Elasticsearch("http://localhost:9200")
gold = pd.DataFrame(d["_source"] for d in helpers.scan(es, index="arxiv-papers-enriched", query={"query": {"match_all": {}}}, size=2000))
arx, hn, cit = read_silver("arxiv-clean"), read_silver("hn-clean"), read_silver("citations-clean")
RESULTS = []
def check(name, cond, detail=""):
    RESULTS.append({"test": name, "resultat": "OK" if bool(cond) else "ECHEC", "detail": detail}); return bool(cond)

## 1. Coherence entre les couches

In [ ]:
n_papers = es.count(index="arxiv-papers")["count"]; n_gold = es.count(index="arxiv-papers-enriched")["count"]
check("Silver ArXiv == index arxiv-papers", len(arx) == n_papers, f"{len(arx):,} vs {n_papers:,}")
check("Silver ArXiv == index arxiv-papers-enriched", len(arx) == n_gold, f"{len(arx):,} vs {n_gold:,}")
check("arxiv_id unique dans le Gold", gold.arxiv_id.is_unique, f"{gold.arxiv_id.duplicated().sum()} doublons")
check("Aucun article perdu ou invente par les jointures", set(gold.arxiv_id) == set(arx.arxiv_id))
check("Couverture citations coherente", gold.has_citation_data.sum() == arx.arxiv_id.isin(cit.arxiv_id).sum(),
      f"{gold.has_citation_data.sum():,} dans le Gold vs {arx.arxiv_id.isin(cit.arxiv_id).sum():,} attendus")
check("Couverture HN coherente", gold.has_hn.sum() == arx.arxiv_id.isin(hn.arxiv_id.dropna()).sum(),
      f"{gold.has_hn.sum():,} dans le Gold vs {arx.arxiv_id.isin(hn.arxiv_id.dropna()).sum():,} attendus")
expected_posts = hn[hn.arxiv_id.isin(arx.arxiv_id)].groupby("arxiv_id").size()
got = gold.set_index("arxiv_id").loc[expected_posts.index, "hn_n_posts"]
check("Nombre de posts HN par article exact", (got == expected_posts).all())
pd.DataFrame(RESULTS)

## 2. Conformite des champs calcules

In [ ]:
check("impact_score dans [0, 100]", gold.impact_score.between(0, 100).all(), f"min={gold.impact_score.min()}, max={gold.impact_score.max()}")
recomputed = 100 * (0.5 * gold.citation_component + 0.3 * gold.hn_component + 0.2 * gold.recency_component)
check("impact_score = formule documentee (a 0,5 pres, arrondis)", (recomputed - gold.impact_score).abs().max() < 0.5, f"ecart max {(recomputed - gold.impact_score).abs().max():.3f}")
check("Composantes dans [0, 1]", gold[["citation_component", "hn_component", "recency_component"]].apply(lambda c: c.between(0, 1).all()).all())
check("Sans donnee HN => hn_component = 0", (gold.loc[~gold.has_hn, "hn_component"] == 0).all())
check("Sans donnee de citation => citation_component = 0", (gold.loc[~gold.has_citation_data, "citation_component"] == 0).all())
check("Rang d'impact : un rang 1 par categorie", (gold.groupby("primary_category").impact_rank_in_category.min() == 1).all())
check("top_terms renseigne", gold.top_terms.map(lambda t: isinstance(t, list) and len(t) > 0).mean() > 0.99, f"{gold.top_terms.map(lambda t: isinstance(t, list) and len(t) > 0).mean()*100:.1f} %")
is_new = gold.arxiv_id.str.match(r"^\d{4}\.")
yr_from_id = 2000 + gold.loc[is_new, "arxiv_id"].str[:2].astype(int)
check("Annee coherente avec l'identifiant ArXiv (>= 97 %)", (yr_from_id == gold.loc[is_new, "year"]).mean() >= 0.97, f"{(yr_from_id == gold.loc[is_new, 'year']).mean()*100:.1f} %")
mp = es.indices.get_mapping(index="arxiv-papers-enriched")["arxiv-papers-enriched"]["mappings"]["properties"]
check("Mapping explicite (pas de mapping dynamique)", mp["title"]["type"] == "text" and mp["primary_category"]["type"] == "keyword" and mp["impact_score"]["type"] == "float")
pd.DataFrame(RESULTS)

## 3. Stabilite : le score se generalise-t-il ?

On separe le corpus en deux moities **disjointes** (parite d'un hachage de `arxiv_id`) et on compare leurs distributions et leurs correlations. Un score robuste doit donner des resultats comparables sur les deux.

In [ ]:
import zlib
from scipy.stats import ks_2samp, spearmanr
gold["half"] = gold.arxiv_id.map(lambda s: zlib.crc32(s.encode()) % 2)
a, b = gold[gold.half == 0], gold[gold.half == 1]
ks = ks_2samp(a.impact_score, b.impact_score)
check("Distribution du score identique sur les deux moities (KS p > 0,01)", ks.pvalue > 0.01, f"KS={ks.statistic:.4f}, p={ks.pvalue:.3f}")
ra = spearmanr(a[a.has_citation_data].cited_by_count, a[a.has_citation_data].impact_score)[0]
rb = spearmanr(b[b.has_citation_data].cited_by_count, b[b.has_citation_data].impact_score)[0]
check("Correlation score x citations stable (ecart < 0,05)", abs(ra - rb) < 0.05, f"moitie A = {ra:.3f}, moitie B = {rb:.3f}")
display(pd.DataFrame({"moitie A": a.impact_score.describe(), "moitie B": b.impact_score.describe()}).round(2))
pd.DataFrame(RESULTS)

## 4. Verification manuelle par echantillon et recherche plein texte

In [ ]:
sample = hn[hn.arxiv_id.isin(arx.arxiv_id)].sample(min(6, hn.arxiv_id.isin(arx.arxiv_id).sum()), random_state=7)
print("Posts HN -> article ArXiv rattache (a verifier a l'oeil : le titre du post doit decrire l'article) :\n")
for _, r in sample.iterrows():
    t = arx.loc[arx.arxiv_id == r.arxiv_id, "title"].iloc[0]
    print(f"HN  : {r.title[:90]}\n  URL : {r.url}\n  ArXiv {r.arxiv_id} : {t[:90]}\n")
res = es.search(index="arxiv-papers", query={"match": {"abstract": "diffusion models"}}, size=5)
titles = [h["_source"]["title"] for h in res["hits"]["hits"]]
check("Recherche plein texte pertinente (`diffusion models`)", res["hits"]["total"]["value"] > 0, f"{res['hits']['total']['value']:,} resultats, took={res['took']} ms")
for t in titles: print("  -", t[:100])
pd.DataFrame(RESULTS)

## 5. Idempotence (procedure manuelle)

1. Noter les comptes ci-dessus. 2. Relancer `31_gold_spark` (ou `99_pipeline_full`). 3. Re-executer ce notebook : les comptes et la distribution du score doivent etre **identiques** (les index sont recrees, l'identifiant de document est `arxiv_id`, aucune duplication possible).

## Bilan

In [ ]:
out = pd.DataFrame(RESULTS); display(out)
print(f"\n{(out.resultat == 'OK').sum()}/{len(out)} tests reussis")